<a href="https://www.kaggle.com/code/thisisrick25/imdb-sentiment-naive-bayes-vs-logistic-regression?scriptVersionId=355207887" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# IMDb Sentiment Classification: Multinomial Naive Bayes vs Logistic Regression

This notebook builds a sentiment classifier using **Multinomial Naive Bayes** and compares its behaviour with **Logistic Regression** using the same IMDb dataset and the same train/test split and TF-IDF representation used in Q2.

It addresses parts **(a)–(g)** of the assignment.

## Assignment Requirements

- **(a)** Use the same training/testing split and TF-IDF representation as Q2.
- **(b)** Train a Multinomial Naive Bayes classifier.
- **(c)** Report accuracy, precision, recall, F1-score and display its confusion matrix.
- **(d)** Train another Multinomial Naive Bayes model using Bag-of-Words.
- **(e)** Compare Naive Bayes (BoW), Naive Bayes (TF-IDF), and Logistic Regression.
- **(f)** Identify ten reviews where Logistic Regression and Naive Bayes disagree.
- **(g)** Report predicted-class probabilities for both models and investigate at least three disagreement cases.

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay,
    confusion_matrix
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print("Libraries imported.")

## 1. Load the IMDb Dataset / Reuse Preprocessed Reviews

The notebook first checks whether the preprocessed dataframe from the previous IMDb work is already available.

If not, it searches `/kaggle/input/` for `IMDB Dataset.csv` and recreates the same preprocessing pipeline used previously.

In [ ]:
if "df" not in globals() or not {"preprocessed_review", "sentiment"}.issubset(df.columns):
    dataset_path = None

    for root, dirs, files in os.walk("/kaggle/input"):
        if "IMDB Dataset.csv" in files:
            dataset_path = os.path.join(root, "IMDB Dataset.csv")
            break

    if dataset_path is None:
        raise FileNotFoundError(
            "IMDB Dataset.csv was not found. Add the IMDb Dataset of 50K Movie Reviews "
            "through Kaggle Notebook -> Add Input."
        )

    print("Dataset:", dataset_path)
    df = pd.read_csv(dataset_path)

    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer

    nltk.download("stopwords", quiet=True)
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)

    stop_words = set(stopwords.words("english"))
    lemmatizer = WordNetLemmatizer()

    def preprocess_review(review):
        text = str(review).lower()
        text = re.sub(r"<[^>]+>", " ", text)
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(r"[^a-z\s]", " ", text)
        tokens = text.split()
        tokens = [word for word in tokens if word not in stop_words]
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
        return " ".join(tokens)

    df["preprocessed_review"] = df["review"].apply(preprocess_review)

else:
    print("Using the existing preprocessed IMDb dataframe.")

df = df.dropna(subset=["review", "preprocessed_review", "sentiment"]).copy()

df["sentiment_num"] = (
    df["sentiment"]
    .astype(str)
    .str.lower()
    .map({"negative": 0, "positive": 1})
)

if df["sentiment_num"].isna().any():
    raise ValueError("Unexpected sentiment labels found.")

print("Dataset shape:", df.shape)

## 2. Same Train/Test Split as Q2

The split uses the same settings as Q2:

- 80% training
- 20% testing
- stratified by sentiment
- `random_state = 42`

Keeping these settings identical makes the model comparison fair.

In [ ]:
X_text = df["preprocessed_review"]
X_raw = df["review"]
y = df["sentiment_num"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train_raw = X_raw.loc[X_train_text.index]
X_test_raw = X_raw.loc[X_test_text.index]

print("Training samples:", len(X_train_text))
print("Testing samples :", len(X_test_text))

## 3. TF-IDF Representation from Q2

Q2 used:

- unigram TF-IDF
- unigram + bigram TF-IDF

For this assignment, **Q2's TF-IDF representation is reused as the primary Naive Bayes representation**.

The vectorizer is fitted only on the training data.

In [ ]:
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 1))

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)

print("TF-IDF training shape:", X_train_tfidf.shape)
print("TF-IDF testing shape :", X_test_tfidf.shape)

# (b) Multinomial Naive Bayes with TF-IDF

`MultinomialNB` can operate on non-negative TF-IDF values, so the same TF-IDF representation can be used for this experiment.

In [ ]:
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)

pred_nb_tfidf = nb_tfidf.predict(X_test_tfidf)

nb_tfidf_metrics = {
    "Accuracy": accuracy_score(y_test, pred_nb_tfidf),
    "Precision": precision_score(y_test, pred_nb_tfidf, zero_division=0),
    "Recall": recall_score(y_test, pred_nb_tfidf, zero_division=0),
    "F1 Score": f1_score(y_test, pred_nb_tfidf, zero_division=0)
}

pd.DataFrame([nb_tfidf_metrics], index=["Naive Bayes - TF-IDF"])

# (c) Naive Bayes Performance and Confusion Matrix

In [ ]:
print("Naive Bayes using TF-IDF")
print(f"Accuracy : {nb_tfidf_metrics['Accuracy']:.4f}")
print(f"Precision: {nb_tfidf_metrics['Precision']:.4f}")
print(f"Recall   : {nb_tfidf_metrics['Recall']:.4f}")
print(f"F1 Score : {nb_tfidf_metrics['F1 Score']:.4f}")

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred_nb_tfidf,
    display_labels=["Negative", "Positive"],
    cmap="Blues",
    ax=ax
)
ax.set_title("Naive Bayes Confusion Matrix — TF-IDF")
plt.show()

# (d) Multinomial Naive Bayes with Bag-of-Words

A separate `CountVectorizer` is used to create Bag-of-Words features.

The BoW vectorizer is also fitted only on the training data.

In [ ]:
bow_vectorizer = CountVectorizer(ngram_range=(1, 1))

X_train_bow = bow_vectorizer.fit_transform(X_train_text)
X_test_bow = bow_vectorizer.transform(X_test_text)

print("BoW training shape:", X_train_bow.shape)
print("BoW testing shape :", X_test_bow.shape)

nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)

pred_nb_bow = nb_bow.predict(X_test_bow)

nb_bow_metrics = {
    "Accuracy": accuracy_score(y_test, pred_nb_bow),
    "Precision": precision_score(y_test, pred_nb_bow, zero_division=0),
    "Recall": recall_score(y_test, pred_nb_bow, zero_division=0),
    "F1 Score": f1_score(y_test, pred_nb_bow, zero_division=0)
}

pd.DataFrame([nb_bow_metrics], index=["Naive Bayes - BoW"])

# (e) Compare Naive Bayes with Logistic Regression

To make the comparison with Q2 consistent, Logistic Regression is trained on the **same TF-IDF representation** and the same train/test split.

The comparison includes:

- Naive Bayes + BoW
- Naive Bayes + TF-IDF
- Logistic Regression + TF-IDF

In [ ]:
# Logistic Regression using the same TF-IDF representation and split as Q2.
logreg = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)
logreg.fit(X_train_tfidf, y_train)

pred_logreg = logreg.predict(X_test_tfidf)

logreg_metrics = {
    "Accuracy": accuracy_score(y_test, pred_logreg),
    "Precision": precision_score(y_test, pred_logreg, zero_division=0),
    "Recall": recall_score(y_test, pred_logreg, zero_division=0),
    "F1 Score": f1_score(y_test, pred_logreg, zero_division=0)
}

In [ ]:
comparison_df = pd.DataFrame([
    {
        "Model": "Naive Bayes",
        "Features": "BoW",
        **nb_bow_metrics
    },
    {
        "Model": "Naive Bayes",
        "Features": "TF-IDF",
        **nb_tfidf_metrics
    },
    {
        "Model": "Logistic Regression",
        "Features": "TF-IDF",
        **logreg_metrics
    }
])

comparison_df

### Comparison

The table above provides the required comparison. The strongest model can be identified by comparing the test-set F1-score, while accuracy, precision, and recall provide additional context.

# (f) Ten Reviews Where Logistic Regression and Naive Bayes Disagree

For the disagreement analysis, Logistic Regression and Naive Bayes use the **same TF-IDF representation**. This isolates differences caused by the learning algorithms rather than by different feature representations.

We identify ten test reviews where their predicted labels differ.

In [ ]:
pred_nb_tfidf = np.asarray(pred_nb_tfidf)
pred_logreg = np.asarray(pred_logreg)

disagreement_mask = pred_nb_tfidf != pred_logreg
disagreement_positions = np.flatnonzero(disagreement_mask)

print("Number of disagreements:", len(disagreement_positions))

if len(disagreement_positions) < 10:
    print("Fewer than 10 disagreements were found for this split.")
    selected_disagreements = disagreement_positions
else:
    # Select ten disagreements spread through the disagreement set.
    selected_disagreements = disagreement_positions[:10]

disagreement_rows = []

for pos in selected_disagreements:
    idx = X_test_text.index[pos]

    nb_pred = int(pred_nb_tfidf[pos])
    lr_pred = int(pred_logreg[pos])

    nb_probs = nb_tfidf.predict_proba(X_test_tfidf[pos])[0]
    lr_probs = logreg.predict_proba(X_test_tfidf[pos])[0]

    disagreement_rows.append({
        "index": idx,
        "actual": "positive" if y_test.iloc[pos] == 1 else "negative",
        "Naive Bayes prediction": "positive" if nb_pred == 1 else "negative",
        "Naive Bayes probability": nb_probs[nb_pred],
        "Logistic Regression prediction": "positive" if lr_pred == 1 else "negative",
        "Logistic Regression probability": lr_probs[lr_pred],
        "review": X_test_raw.loc[idx]
    })

disagreement_df = pd.DataFrame(disagreement_rows)
display(disagreement_df)

# (g) Predicted-Class Probabilities

For each disagreement case, the probability shown for each model is the probability assigned to **that model's own predicted class**.

This allows us to distinguish between:

- strong disagreement, where one model is highly confident;
- borderline disagreement, where one or both models are uncertain.

In [ ]:
# Add probability margin information for easier comparison.
if len(disagreement_df) > 0:
    disagreement_df["NB confidence"] = disagreement_df["Naive Bayes probability"]
    disagreement_df["LR confidence"] = disagreement_df["Logistic Regression probability"]

display(disagreement_df[[
    "index",
    "actual",
    "Naive Bayes prediction",
    "Naive Bayes probability",
    "Logistic Regression prediction",
    "Logistic Regression probability",
    "review"
]])

## Detailed Investigation of Three Disagreement Cases

The following analysis selects three disagreement cases.

For each case, we inspect the TF-IDF features that occur in the review and compare how the two models respond to the same words/features.

This helps explain why Naive Bayes and Logistic Regression can produce different predictions even when they receive exactly the same TF-IDF input.

In [ ]:
# Inspect three disagreement cases in detail.
n_cases = min(3, len(selected_disagreements))

for case_num, pos in enumerate(selected_disagreements[:n_cases], start=1):
    idx = X_test_text.index[pos]
    text = X_test_text.loc[idx]
    raw_review = X_test_raw.loc[idx]

    nb_pred = int(pred_nb_tfidf[pos])
    lr_pred = int(pred_logreg[pos])

    nb_probs = nb_tfidf.predict_proba(X_test_tfidf[pos])[0]
    lr_probs = logreg.predict_proba(X_test_tfidf[pos])[0]

    # Obtain non-zero TF-IDF features for this review.
    row = X_test_tfidf[pos]
    feature_names = np.array(tfidf_vectorizer.get_feature_names_out())

    present_indices = row.indices
    tfidf_values = row.data

    # Logistic Regression contribution: TF-IDF * coefficient.
    lr_coefficients = logreg.coef_.ravel()
    lr_contributions = []

    # Naive Bayes log-count difference gives a useful directional
    # feature signal for explaining its class preference.
    nb_log_ratio = (
        nb_tfidf.feature_log_prob_[1] -
        nb_tfidf.feature_log_prob_[0]
    )

    nb_contributions = []

    for feature_idx, tfidf_value in zip(present_indices, tfidf_values):
        feature = feature_names[feature_idx]

        lr_contribution = tfidf_value * lr_coefficients[feature_idx]
        nb_directional_contribution = (
            tfidf_value * nb_log_ratio[feature_idx]
        )

        lr_contributions.append(
            (feature, tfidf_value, lr_contribution)
        )
        nb_contributions.append(
            (feature, tfidf_value, nb_directional_contribution)
        )

    lr_contributions.sort(key=lambda x: abs(x[2]), reverse=True)
    nb_contributions.sort(key=lambda x: abs(x[2]), reverse=True)

    print("=" * 90)
    print(f"CASE {case_num}")
    print(f"Index: {idx}")
    print(f"Actual sentiment: {'positive' if y_test.iloc[pos] == 1 else 'negative'}")
    print(f"Naive Bayes prediction: {'positive' if nb_pred == 1 else 'negative'} "
          f"(probability={nb_probs[nb_pred]:.4f})")
    print(f"Logistic Regression prediction: {'positive' if lr_pred == 1 else 'negative'} "
          f"(probability={lr_probs[lr_pred]:.4f})")

    print("\nOriginal review:")
    print(raw_review)

    print("\nTop Logistic Regression feature contributions:")
    for feature, tfidf_value, contribution in lr_contributions[:10]:
        print(
            f"  {feature:<25} TF-IDF={tfidf_value:.4f} "
            f"contribution={contribution:+.4f}"
        )

    print("\nTop Naive Bayes directional feature contributions:")
    for feature, tfidf_value, contribution in nb_contributions[:10]:
        print(
            f"  {feature:<25} TF-IDF={tfidf_value:.4f} "
            f"directional contribution={contribution:+.4f}"
        )

## Interpretation of the Three Cases

When examining the three cases above, consider:

- **Word frequency:** Naive Bayes models class-specific word likelihoods, so frequently occurring sentiment words can strongly affect its prediction.
- **Feature weights:** Logistic Regression learns discriminative coefficients that indicate how individual features move the decision boundary.
- **TF-IDF weighting:** A feature can have a high TF-IDF value in a particular review even if it is not simply a frequent word.
- **Negation:** Both models may struggle with expressions such as “not good” because individual words can have sentiment associations that conflict with the sentence meaning.
- **Mixed sentiment:** Reviews often contain praise and criticism simultaneously.
- **Context:** The same word can express different sentiment depending on the surrounding text.
- **Model assumptions:** Naive Bayes assumes conditional independence of features, while Logistic Regression learns a discriminative linear decision boundary.

Therefore, disagreements are useful for understanding how the two models use the same feature representation differently.

# Final Summary

This notebook completed the requested Multinomial Naive Bayes experiment:

- Used the same 80/20 stratified split and TF-IDF representation as Q2.
- Trained Multinomial Naive Bayes using TF-IDF.
- Reported accuracy, precision, recall, and F1-score.
- Displayed the Naive Bayes TF-IDF confusion matrix.
- Trained a second Naive Bayes model using Bag-of-Words.
- Compared Naive Bayes (BoW), Naive Bayes (TF-IDF), and Logistic Regression (TF-IDF).
- Identified ten reviews where Logistic Regression and Naive Bayes disagree.
- Reported the probability assigned to each model's predicted class.
- Investigated three disagreement cases using the words/features contributing to each model's prediction.